# 02 Metrics, Robustness, And Self-Healing Visualisations

Here I stress the scalar field instead of tuning it. I use obstruction and recovery plots to check whether the Bessel zone behaves like a self-healing beam and whether the metric traces tell the same story as the images.

In [1]:
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from vbb_study import setup_study, vbb_style

PATHS = setup_study.bootstrap(Path.cwd())
ROOT = PATHS["root"]
PUB_ROOT = PATHS["publication"]
LAB_ROOT = PATHS["modular_lab"]

import bessel_twin_core as bt
from Publication_Study import publication_diagnostics as pdiag

In [2]:
PRESET = "fast"
PUB_OUT = pdiag.publication_output_tree(PATHS["outputs"])
base = bt.default_config(PRESET)

## Thin-Screen Obstacle Campaign

The campaign uses a baseline ell = 3, core = 3 um, length = 150 um case and compares a few obstacle types at the same obstacle plane depth.

In [3]:
campaign = [
    {"case_id": "disk_small", "obstacle_kind": "disk", "obstacle_radius_m": 1.5 * bt.um},
    {"case_id": "disk_large", "obstacle_kind": "disk", "obstacle_radius_m": 2.5 * bt.um},
    {"case_id": "strip", "obstacle_kind": "strip", "obstacle_width_m": 2.0 * bt.um},
    {"case_id": "square", "obstacle_kind": "square", "obstacle_width_m": 2.5 * bt.um, "obstacle_height_m": 2.5 * bt.um},
]
ideal_bundles = [
    pdiag.build_self_healing_bundle(base, preset=PRESET, path="ideal", **spec)
    for spec in campaign
]
ideal_summary = pdiag.self_healing_summary_dataframe(ideal_bundles)
ideal_summary.to_csv(PUB_OUT["csv"] / "self_healing_campaign_ideal.csv", index=False)
display(ideal_summary)

,case_id,path,obstacle_kind,obstacle_z_um,blocked_fraction,peak_recovery_end,peak_recovery_max,peak_recovery_mean,onaxis_recovery_end,target_bessel_length_um,ell
0,disk_small,ideal,disk,52.5,0.000427,0.999752,1.127699,0.998717,0.630029,150.0,3
1,disk_large,ideal,disk,52.5,0.001205,1.002520,1.011374,0.921472,3.688061,150.0,3
2,strip,ideal,strip,52.5,0.015625,1.531466,1.531466,1.212772,144.941013,150.0,3
3,square,ideal,square,52.5,0.000381,1.004413,1.189859,1.011897,0.600651,150.0,3


## Selected Diagnostic Panel

Inspect one campaign member in detail: reference field at the obstacle, obstructed field, obstructed XZ evolution, and recovery traces.

In [4]:
SELECTED = "disk_large"
selected = next(bundle for bundle in ideal_bundles if bundle["case_id"] == SELECTED)
fig = pdiag.plot_self_healing_diagnostics(selected)
selected_path = PUB_OUT["figures"] / f"{SELECTED}_self_healing_panel.png"
fig.savefig(selected_path, dpi=220, bbox_inches="tight")
plt.show()
print(selected_path)

C:\PhD\Code\Publication_Study\Publication_Study\outputs\figures\publication_study\disk_large_self_healing_panel.png


C:\Users\sm2006\AppData\Local\Temp\ipykernel_21076\599695180.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Recovery Curve Comparison

Compare how the peak-in-plane recovery evolves for all ideal campaign members after the obstacle plane.

In [5]:
fig, ax = plt.subplots(figsize=(8, 4.8), constrained_layout=True)
for bundle in ideal_bundles:
    ax.plot(bundle["z_relative"] / bt.um, bundle["peak_recovery"], label=bundle["case_id"])
ax.axhline(1.0, color="0.5", linewidth=0.8)
ax.set_title("Peak recovery after obstacle")
ax.set_xlabel("z after obstacle [um]")
ax.set_ylabel("obstructed / reference peak")
ax.legend(frameon=False)
curve_path = PUB_OUT["figures"] / "self_healing_recovery_curves.png"
fig.savefig(curve_path, dpi=220, bbox_inches="tight")
plt.show()
print(curve_path)

C:\PhD\Code\Publication_Study\Publication_Study\outputs\figures\publication_study\self_healing_recovery_curves.png


C:\Users\sm2006\AppData\Local\Temp\ipykernel_21076\877199825.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Realistic Benchmark

Repeat one obstruction with the realistic device-to-sample path so the paper can show that the robustness diagnostic layer also works on the lab-facing propagation route.

In [6]:
realistic_bundle = pdiag.build_self_healing_bundle(
    base,
    preset=PRESET,
    path="realistic",
    case_id="disk_large_realistic",
    obstacle_kind="disk",
    obstacle_radius_m=2.5 * bt.um,
)
compare = pd.DataFrame([selected["metrics"], realistic_bundle["metrics"]])
compare.to_csv(PUB_OUT["csv"] / "self_healing_realistic_benchmark.csv", index=False)
display(compare)

,case_id,path,obstacle_kind,obstacle_z_um,blocked_fraction,peak_recovery_end,peak_recovery_max,peak_recovery_mean,onaxis_recovery_end,target_bessel_length_um,ell
0,disk_large,ideal,disk,52.5,0.001205,1.002520,1.011374,0.921472,3.688061,150.0,3
1,disk_large_realistic,realistic,disk,52.5,0.001205,0.999782,1.014166,0.998742,0.970763,150.0,3


In [7]:
fig = pdiag.plot_self_healing_diagnostics(realistic_bundle, title="Realistic device-path self-healing benchmark")
realistic_path = PUB_OUT["figures"] / "self_healing_realistic_benchmark.png"
fig.savefig(realistic_path, dpi=220, bbox_inches="tight")
plt.show()
print(realistic_path)

C:\PhD\Code\Publication_Study\Publication_Study\outputs\figures\publication_study\self_healing_realistic_benchmark.png


C:\Users\sm2006\AppData\Local\Temp\ipykernel_21076\1538967723.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
